# Disaggregated error analysis with Fairlearn

Synthetic scenario: a support-ticket escalation classifier is used across two interaction languages. The data is intentionally constructed with poorer recall for one group so aggregate accuracy cannot hide it. This is a measurement lab, not a legal conclusion.

In [ ]:
from pathlib import Path
import importlib.util
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
rng = np.random.default_rng(17)
n_per_group = 240
language = np.array(["English"] * n_per_group + ["Hindi"] * n_per_group)
y_true = rng.binomial(1, 0.40, size=2 * n_per_group)
y_pred = y_true.copy()

# English: roughly 8% random errors.
english_idx = np.where(language == "English")[0]
flip_en = rng.choice(english_idx, size=20, replace=False)
y_pred[flip_en] = 1 - y_pred[flip_en]

# Hindi: deliberately higher false-negative rate plus some false positives.
hindi_idx = np.where(language == "Hindi")[0]
hindi_pos = hindi_idx[y_true[hindi_idx] == 1]
hindi_neg = hindi_idx[y_true[hindi_idx] == 0]
missed = rng.choice(hindi_pos, size=max(1, int(0.38 * len(hindi_pos))), replace=False)
false_alarms = rng.choice(hindi_neg, size=max(1, int(0.12 * len(hindi_neg))), replace=False)
y_pred[missed] = 0
y_pred[false_alarms] = 1

df = pd.DataFrame({"language": language, "y_true": y_true, "y_pred": y_pred})
df.groupby("language").size().rename("count")

In [ ]:
def selection_rate(y_true, y_pred):
    return float(np.mean(np.asarray(y_pred) == 1))

def false_positive_rate(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mask = y_true == 0
    return float(np.mean(y_pred[mask] == 1)) if mask.any() else np.nan

def false_negative_rate(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mask = y_true == 1
    return float(np.mean(y_pred[mask] == 0)) if mask.any() else np.nan

metrics = {
    "accuracy": accuracy_score,
    "selection_rate": selection_rate,
    "false_positive_rate": false_positive_rate,
    "false_negative_rate": false_negative_rate,
}

## Compute the same metrics through Fairlearn's `MetricFrame`

`by_group` makes disaggregation explicit; `difference()` and `ratio()` summarize disparity but must be read alongside the underlying rows, base rates, sample sizes, and uncertainty.

In [ ]:
fairlearn_installed = importlib.util.find_spec("fairlearn") is not None
if fairlearn_installed:
    from fairlearn.metrics import MetricFrame
    frame = MetricFrame(
        metrics=metrics,
        y_true=df["y_true"],
        y_pred=df["y_pred"],
        sensitive_features=df["language"],
    )
    by_group = frame.by_group
    overall = frame.overall
    disparity_difference = frame.difference()
    disparity_ratio = frame.ratio()
else:
    by_group = df.groupby("language").apply(
        lambda g: pd.Series({name: fn(g.y_true, g.y_pred) for name, fn in metrics.items()}),
        include_groups=False,
    )
    overall = pd.Series({name: fn(df.y_true, df.y_pred) for name, fn in metrics.items()})
    disparity_difference = by_group.max() - by_group.min()
    disparity_ratio = by_group.min() / by_group.max().replace(0, np.nan)
    print("Fairlearn is not installed; displaying an equivalent pandas fallback.")

display(pd.DataFrame({"overall": overall}))
display(by_group)
display(pd.DataFrame({"difference": disparity_difference, "ratio": disparity_ratio}))

## Add uncertainty with a transparent bootstrap

Small groups and rare outcomes can produce unstable rates. The bootstrap here is illustrative; choose an uncertainty method appropriate to the metric and sampling design.

In [ ]:
def bootstrap_metric(group_df, metric_fn, rounds=500, seed=23):
    local_rng = np.random.default_rng(seed)
    values = []
    for _ in range(rounds):
        sample_idx = local_rng.integers(0, len(group_df), len(group_df))
        sample = group_df.iloc[sample_idx]
        values.append(metric_fn(sample.y_true, sample.y_pred))
    return np.quantile(values, [0.025, 0.5, 0.975])

intervals = []
for group, group_df in df.groupby("language"):
    lo, mid, hi = bootstrap_metric(group_df, false_negative_rate)
    intervals.append({"language": group, "metric": "false_negative_rate", "lower": lo, "median": mid, "upper": hi, "n": len(group_df)})
intervals_df = pd.DataFrame(intervals)
intervals_df

In [ ]:
plot_data = by_group.reset_index()
if plot_data.columns[0] != "language":
    plot_data = plot_data.rename(columns={plot_data.columns[0]: "language"})
ax = plot_data.plot(x="language", y="false_negative_rate", kind="bar", legend=False, title="False-negative rate by interaction language")
ax.set_ylabel("False-negative rate")
plt.tight_layout()
plt.show()

## Decision, not dashboard theater

The synthetic Hindi false-negative rate is materially worse: urgent tickets are more likely to be missed. A reasonable response might be to constrain launch, improve language-specific data/labels, evaluate speech/OCR upstream errors, adjust workflow or thresholds with domain owners, and add human review—then remeasure all error types and utility.

In [ ]:
assessment = {
    "use_case": "synthetic support-ticket escalation",
    "groups": ["English", "Hindi"],
    "sample_sizes": df.groupby("language").size().to_dict(),
    "overall": overall.to_dict(),
    "by_group": by_group.reset_index().to_dict(orient="records"),
    "difference": disparity_difference.to_dict(),
    "ratio": disparity_ratio.to_dict(),
    "uncertainty": intervals_df.to_dict(orient="records"),
    "decision": "Do not ship as one undifferentiated workflow; investigate and mitigate Hindi false negatives.",
    "caveats": [
        "synthetic data", "groups are not exhaustive", "metrics do not determine legality or ethics",
        "deployment workflow and error costs require domain review"
    ],
}
out = Path("_evidence/09_fairness_assessment.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(assessment, indent=2, default=float), encoding="utf-8")
assert by_group.loc["Hindi", "false_negative_rate"] > by_group.loc["English", "false_negative_rate"]
print("PASS: aggregate metrics did not hide the designed subgroup failure")
print("Wrote", out.resolve())